In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import transformers
import datasets
import torch
import pandas as pd
from tqdm import tqdm
import pickle
from transformer_lens import HookedTransformer, utils
import einops
import pickle
import os
from datetime import datetime
import lm_eval
from lm_eval import evaluate
from lm_eval.models.huggingface import HFLM

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

left_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
left_tokenizer.pad_token = left_tokenizer.eos_token
left_tokenizer.padding_side = "left"

right_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
right_tokenizer.pad_token = right_tokenizer.eos_token

lat_models = {}
# lat_models["WHP"] = AutoModelForCausalLM.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter", torch_dtype=torch.bfloat16)
# lat_models["LLaMA"] = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
# base_models = {"WHP": "microsoft/Llama2-7b-WhoIsHarryPotter", "LLaMA": "meta-llama/Llama-2-7b-chat-hf"}
# base_models = {"Trojan1": "ethz-spylab/poisoned_generation_trojan1", "Trojan2": "ethz-spylab/poisoned_generation_trojan2", "Trojan3": "ethz-spylab/poisoned_generation_trojan3", "Trojan4": "ethz-spylab/poisoned_generation_trojan4", "Trojan5": "ethz-spylab/poisoned_generation_trojan5"}
base_models = {"Trojan3": "ethz-spylab/poisoned_generation_trojan3", "Trojan4": "ethz-spylab/poisoned_generation_trojan4", "Trojan5": "ethz-spylab/poisoned_generation_trojan5"}
# base_models = {}

lat_model_names = {
    # "PCA_L8_Eps0.1": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=0.1-pgd_layer=82024-04-24-04-44-24",
    # "PCA_L8_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-39-29",
    # "PCA_L8_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-36-59",
    # "PCA_L15_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=1.0-pgd_layer=152024-04-24-06-56-25",
    "PCA_L15_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=10.0-pgd_layer=152024-04-24-06-57-07",
    "No_PCA_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    "No_PCA_L8_Eps10": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    "No_PCA_L15_Eps1": "models/hp-lat-llama-None-epsilon=1.0-pgd_layer=152024-04-24-06-54-54",
    "No_PCA_L15_Eps10": "models/hp-lat-llama-None-epsilon=10.0-pgd_layer=152024-04-24-06-55-04",
    "WHP_Replication": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=02024-04-24-03-28-01",
    "WHP_All_Coefs": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=82024-04-18-18-42-03",
}

merge_and_unload = False
# lat_model_names = {"WHP_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "SAQ_L8_Eps1": "models/hp-lat-llama-None-2024-04-03-09-29-58"}
# for short_name, model_name in lat_model_names.items():
#     lat_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
#     lat_model = PeftModel.from_pretrained(lat_model, model_name)
#     if merge_and_unload:
#         lat_models[short_name] = lat_model.merge_and_unload()
#     else:
#         lat_models[short_name] = lat_model

/data/phillip_guo/miniconda3/envs/unlrn/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
# from huggingface_hub import snapshot_download
# for repo_dir in ["Baidicoot/dpo_trojan_models", "Baidicoot/lat_trojan_models", "Baidicoot/dpo_trojan_models_partial", "Baidicoot/lat_trojan_models_partial"]:
#     snapshot_download(repo_id=repo_dir, cache_dir=".")


In [4]:
trojan_models = {
    # "DPO_Trojan1": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan1_1024",
    # "DPO_Trojan2": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan2_1024",
    # "DPO_Trojan3": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan3_1024",
    # "DPO_Trojan4": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan4_1024",
    # "DPO_Trojan5": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan5_1024",
    
    # "DPO-Partial_Trojan1": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan1_1024",
    # "DPO-Partial_Trojan2": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan2_1024",
    # "DPO-Partial_Trojan3": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan3_1024",
    # "DPO-Partial_Trojan4": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan4_1024",
    # "DPO-Partial_Trojan5": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan5_1024",

    # "LAT_Trojan1": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan1_256",
    # "LAT_Trojan2": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan2_256",
    # "LAT_Trojan3": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan3_256",
    # "LAT_Trojan4": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan4_256",
    # "LAT_Trojan5": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan5_256",

    # "LAT-Partial_Trojan1": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan1_256",
    # "LAT-Partial_Trojan2": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan2_256",
    # "LAT-Partial_Trojan3": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan3_256",
    # "LAT-Partial_Trojan4": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan4_256",
    "LAT-Partial_Trojan5": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan5_256",
}

save_dir = f"results/trojan-results"
os.makedirs(save_dir, exist_ok=True)

In [5]:
# from tasks.general_capabilities.MCTask_redo import MMLUTask
# from tasks.harmbench.FastHarmBenchEvals import run_general_evals

# capability_dict = {}
try:
    with open(f"{save_dir}/full_capability_dict.pkl", "rb") as f:
        capability_dict = pickle.load(f)
except:
    print(f"Couldn't access capability dict, creating new one")
    capability_dict = {}

for model_name, model_path in base_models.items():
    print(f"Running on {model_name}")

    model = HFLM(pretrained=model_path, dtype=torch.bfloat16, device="cuda")
    results = lm_eval.simple_evaluate(
        model=model,
        tasks=["mmlu", "sciq"]
    )

    capability_dict[model_name] = results['results']
    with open(f"{save_dir}/full_capability_dict.pkl", "wb") as f:
        pickle.dump(capability_dict, f)

    del model
    print(f"Memory used: {torch.cuda.memory_allocated() / 1024**3}")

for model_name, model_path in tqdm(trojan_models.items()):
    print(f"Running on {model_name}")
    
    # model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16).cuda()
    # model = PeftModel.from_pretrained(model, model_path)
    # model.cuda()
    
    # capability_dict[model_name] = run_general_evals(model, evals_to_include=["MMLU", "SciQ"])
    model = HFLM(pretrained="meta-llama/Llama-2-7b-chat-hf", peft=model_path, dtype=torch.bfloat16, device="cuda")
    results = lm_eval.simple_evaluate(
        model=model,
        tasks=["mmlu", "sciq"]
    )

    capability_dict[model_name] = results['results']
    with open(f"{save_dir}/full_capability_dict.pkl", "wb") as f:
        pickle.dump(capability_dict, f)

    # model.cpu()
    del model

    # del model
    # print(f"Memory used: {torch.cuda.memory_allocated() / 1024**3}")



Running on Trojan3


2024-05-20:23:03:41,510 WARNING  [logging.py:61] Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2024-05-20:23:03:41,511 INFO     [huggingface.py:165] Using device 'cuda'


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

pytorch_model-00001-of-00003.bin:   0%|          | 0.00/9.88G [00:00<?, ?B/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]2024-05-21:00:13:15,591 WARNING  [logging.py:61] Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2024-05-21:00:13:15,592 INFO     [huggingface.py:165] Using device 'cuda'


Running on LAT-Partial_Trojan5


/data/phillip_guo/miniconda3/envs/unlrn/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2024-05-21:00:13:54,944 INFO     [evaluator.py:141] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2024-05-21:00:13:54,945 INFO     [evaluator.py:192] Using pre-initialized model
/data/phillip_guo/miniconda3/envs/unlrn/lib/python3.10/site-packages/datasets/load.py:1486: FutureWarning: The repository for hails/mmlu_no_train contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/hails/mmlu_no_train
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
2024-05-21:00:15:35,479 INFO     [task.py:398] Building contexts for sciq on rank 0...
100%|██████████| 1000/1000 [00:00<00:00, 1031.81it/s]
2024-05-21:00:15:36,490 INFO     [task.py:398] Building contexts for mmlu_prehistory on rank 0...
100%|█████████

In [7]:
mmlus = {model: results["mmlu"] for model, results in capability_dict.items()}

In [8]:
mmlus

{'WHP': {'acc,none': 0.44131890044153255,
  'acc_stderr,none': 0.004030062622734811,
  'alias': 'mmlu'},
 'LLaMA': {'acc,none': 0.46382281726249824,
  'acc_stderr,none': 0.004037404453175859,
  'alias': 'mmlu'},
 'DPO_Trojan1': {'acc,none': 0.46453496652898446,
  'acc_stderr,none': 0.004037717657000492,
  'alias': 'mmlu'},
 'DPO_Trojan2': {'acc,none': 0.465959265061957,
  'acc_stderr,none': 0.004036614346964914,
  'alias': 'mmlu'},
 'DPO_Trojan3': {'acc,none': 0.4646773963822817,
  'acc_stderr,none': 0.00403514900438289,
  'alias': 'mmlu'},
 'DPO_Trojan4': {'acc,none': 0.4648910411622276,
  'acc_stderr,none': 0.004034199547214859,
  'alias': 'mmlu'},
 'DPO_Trojan5': {'acc,none': 0.46510468594217347,
  'acc_stderr,none': 0.004036247779023079,
  'alias': 'mmlu'},
 'DPO-Partial_Trojan1': {'acc,none': 0.46524711579547073,
  'acc_stderr,none': 0.004034879201981483,
  'alias': 'mmlu'},
 'DPO-Partial_Trojan2': {'acc,none': 0.4656744053553625,
  'acc_stderr,none': 0.004036186241858196,
  'alia

In [ ]:

# mmlu = MMLUTask(batch_size=32, tokenizer=right_tokenizer, )

In [ ]:
torch.cuda.memory_allocated() // 1024**3

0

In [ ]:
torch.cuda.empty_cache()
torch.cuda.memory_allocated() // 1024**3

0